# DSA 4060 – Recommender Systems
## Week 3 Practical Lab: Building a Simple Content-Based Recommender

**Lab objective:** Build a content-based recommender that goes from
**Movie Features → User Preferences → Recommendation Scores → Top Recommendations**.

I represented movies using structured binary item features (genres), built a
user preference profile from that user's ratings, scored every unseen movie
against the profile, and generated Top-N recommendations. I then tested a second
user, added my own movie and user, and reflected on the model's limitations.

This lab deliberately uses **structured features only**.

## 1. Import Libraries

Only **pandas** is needed for this lab, since all computation is done with
DataFrame operations (merge, multiply, sum, sort).

In [ ]:
import pandas as pd

## 2. Create / Load the Dataset

The dataset is created in-code (as specified in the lab brief). Each row is a
movie described by binary genre features, where **1 = feature present** and
**0 = feature absent**. For example, *Galaxy Mission* has Action = 1 and
SciFi = 1, so its item profile is [1, 0, 0, 0, 1].

| Column | Meaning |
|---|---|
| movie_id | Unique movie identifier |
| title | Movie title |
| Action / Comedy / Drama / Romance / SciFi | Binary structured features (1 = present, 0 = absent) |

In [ ]:
movies = pd.DataFrame({
    "movie_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "title": [
        "Galaxy Mission", "Love in Nairobi", "Final Battle", "Family Weekend",
        "Future World", "Broken Hearts", "City Detectives", "Space Warriors",
        "Wedding Weekend", "Hidden Truth"
    ],
    "Action":  [1, 0, 1, 0, 1, 0, 1, 1, 0, 0],
    "Comedy":  [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    "Drama":   [0, 1, 0, 0, 0, 1, 1, 0, 0, 1],
    "Romance": [0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    "SciFi":   [1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
})
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


## 3. Explore the Dataset

Checked the first rows, dimensions and data types to confirm the data loaded
correctly and all features are numeric.

In [ ]:
movies.head()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


In [ ]:
movies.shape

(10, 7)

In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes


**Answers (Step 5):**

1. There are **10 movies** in the dataset.
2. There are **5 genre features** (Action, Comedy, Drama, Romance, SciFi).
3. **One row represents one movie**, described by its unique ID, title, and a
   binary profile over the five genres (1 = genre present, 0 = absent).

## 4. Identify the Structured Features

Defined the list of feature columns. These are the **structured features**
that describe every movie in a consistent, comparable way. Separating them
from identifiers (movie_id, title) makes later calculations clean.

In [ ]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


## 5. Create Item Profiles

The **item-feature matrix** above *is* the set of item profiles: each row is
one movie's profile vector over the five genres. Let's inspect a single
profile to make the encoding concrete.

In [ ]:
movies.loc[movies["title"] == "Galaxy Mission"]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


**Explanation (Step 7):** For *Galaxy Mission*, the 1s in the **Action** and
**SciFi** columns mean the movie belongs to those two genres, while the 0s in
Comedy, Drama and Romance mean it does not belong to those. Its item profile is
therefore **[1, 0, 0, 0, 1]**.

## 6. Create User Ratings

Now we introduce **User A**, who has rated four movies. These ratings are the
evidence from which we will infer the user's preferences.

In [ ]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


## 7. Join Ratings With Movie Features

Merged the user's ratings with the movie table on `movie_id`, so each rated
movie is attached to its genre feature vector.

In [ ]:
user_movies = ratings.merge(movies, on="movie_id")
user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


## 8. Examine the User's History

In [ ]:
user_movies[["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


**Observation (Step 10):** User A appears to prefer **Action and Sci-Fi**
movies. Their two highest ratings (5 stars) went to *Galaxy Mission* (Action +
SciFi) and *Future World* (Action + SciFi), while their only low rating (2
stars) went to *Love in Nairobi*, a Romance/Drama film with no Action or
SciFi. Even their 4-star rating (*Final Battle*) is an Action movie. So their
history strongly suggests an Action/SciFi preference and a dislike of
Romance/Drama.

## 9. Build the User Profile

**Step 1 — select the feature columns and the ratings** of the movies the user
has rated.

In [ ]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]
rated_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,1,0,0,0,1


In [ ]:
user_ratings

0    5
1    2
2    4
3    5
Name: rating, dtype: int64

**Step 2 — weight features by rating.** Multiplying each movie's feature
vector by its rating means a highly rated movie contributes more strongly to
the user's preference profile, while a low-rated movie contributes little (or
subtracts nothing, but dilutes the profile).

In [ ]:
weighted_features = rated_features.mul(user_ratings, axis=0)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


**Step 3 — sum the weighted features.** The column sums form the user's
**preference profile**: the total (rating-weighted) evidence for each genre.

In [ ]:
user_profile = weighted_features.sum()
user_profile

Action     14
Comedy      0
Drama       2
Romance     2
SciFi      10
dtype: int64

## 10. Interpret the User Profile

In [ ]:
user_profile.sort_values(ascending=False)

Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

**Question (Step 14):** The two strongest genres are **Action (score 14)**
and **SciFi (score 10)**. Action got contributions from all three Action films
the user rated (5 + 4 + 5 stars), and SciFi got contributions from the two
SciFi films they loved (5 + 5 stars). Romance and Drama score only 2 — from the
single 2-star rating — confirming the user's weak preference for them. These
scores summarize which features the user tends to prefer.

## 11. Normalize the User Profile

Normalization divides each score by the profile total, converting the profile
into **proportions** (they sum to 1). This makes the scores easier to interpret
as preference *weights* and keeps recommendation scores on a consistent scale
regardless of how many movies the user rated.

In [ ]:
user_profile_normalized = (user_profile / user_profile.sum())
user_profile_normalized

Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64

## 12. Calculate Recommendation Scores

We score every movie by taking a weighted sum of its features using the
normalized user profile. A movie's score is high when it contains *more of the
features the user prefers* — e.g. an Action+SciFi movie scores almost the full
profile weight, while a Romance/Drama movie scores only the small Romance and
Drama weights.

In [ ]:
movies["recommendation_score"] = (
    movie_features.mul(user_profile_normalized).sum(axis=1)
)
movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


## 13. Rank the Movies

A higher recommendation score means the movie contains more of the features
preferred by the user, so we sort in descending order.

In [ ]:
ranked_movies = movies.sort_values("recommendation_score", ascending=False)
ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


## 14. Remove Watched Movies

A recommender should never suggest a movie the user has already seen, so we
filter out the rated movie IDs before producing the final list.

In [ ]:
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[~ranked_movies["movie_id"].isin(watched_movies)]
recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


## 15. Generate Top-N Recommendations (Top 3)

In [ ]:
top_3 = recommendations[["title", "recommendation_score"]].head(3)
top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


## 16. Explain the Recommendations (Step 20)

1. **Space Warriors (0.857)** — recommended because the user's strongest
   preferences are **Action (0.500)** and **SciFi (0.357)**, and this movie has
   exactly those two features, so it captures almost the entire profile weight.
2. **City Detectives (0.571)** — recommended because it combines the user's
   strongest feature (Action) with a small Drama weight.
3. **Broken Hearts (0.143)** — recommended because it is a Drama + Romance
   movie, which received small but non-zero weights from the 2-star rating of
   *Love in Nairobi*.

Movies already watched (e.g. *Final Battle* with score 0.500, and *Galaxy
Mission* with 0.857) were removed even though some would have scored highly.
Note that *Family Weekend* scores 0.000 — the Comedy genre has zero weight in
this user's profile, so it can never be recommended to User A.

## 17. Build a Reusable Function

All the steps above are wrapped into one function so we can reuse the pipeline
for any user and any number of recommendations. It follows the content-based
workflow: **Item Metadata → Item Profiles → User Ratings → User Profile →
Candidate Scores → Remove Seen Items → Rank → Top-N**.

In [ ]:
def recommend_movies(movies, ratings, features, n=3):
    # Join the user's ratings to the movie features
    user_movies = ratings.merge(movies, on="movie_id")
    rated_features = user_movies[features]
    # Weight features by rating and sum into a user profile
    weighted_features = rated_features.mul(user_movies["rating"], axis=0)
    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()   # normalize
    # Score every movie against the normalized profile
    scores = movies[features].mul(user_profile).sum(axis=1)
    results = movies.copy()
    results["recommendation_score"] = scores
    # Remove movies already watched, rank, and return Top-N
    watched = ratings["movie_id"]
    results = results[~results["movie_id"].isin(watched)]
    return results.sort_values("recommendation_score", ascending=False).head(n)

# Test it on User A
recommend_movies(movies, ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


## 18. Test a Second User (User B)

User B rated: *Love in Nairobi* (5), *Family Weekend* (4), *Broken Hearts* (5),
*Wedding Weekend* (4) — a Romance/Drama/Comedy-oriented viewer.

In [ ]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})
recommend_movies(movies, user_b_ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


**Comparison (Step 22):** Users A and B received **completely different
recommendations**. User A (Action/SciFi viewer) got *Space Warriors*, *City
Detectives* and *Broken Hearts*, while User B (Romance/Drama viewer) got
*Hidden Truth* and *City Detectives* (tied at 0.3125) with *Final Battle*
scoring 0.000. This happens because the user profile is built from **that
user's own ratings**: different ratings produce different profile weights, and
therefore different scores. This personalization is the key advantage of
content-based filtering over last week's popularity baseline.

## 19. Thinking Exercise — User C (Step 23)

Suppose User C has watched only *Galaxy Mission* and rated it 5 stars.

1. **What will the model learn about User C?** Its entire profile comes from a
   single movie, so it learns "this user likes Action and SciFi" — and nothing
   else. There is zero evidence about the other three genres.
2. **Which genres will dominate the profile?** Action and SciFi each get weight
   5/10 = 0.5; Comedy, Drama and Romance get 0.
3. **What could go wrong?** Any Action or SciFi movie — even a bad match for the
   user's actual taste — will outrank every non-Action/non-SciFi movie. The
   model cannot distinguish *why* they liked the film (the action? the story?
   the actors?); it just propagates the genre pattern.
4. **Is one rating enough?** No. One rating gives a very narrow, possibly
   misleading profile. Cold-start users need more interactions, or hybrid /
   demographic fallback strategies, before content-based recommendations become
   reliable.

## 20. Add Your Own Unique Movie (Step 24)

Every student must add one unique movie with their own title and feature
combination. Mine is **"Savannah Run"** — a Comedy/Drama film.

In [ ]:
new_movie = {
    "movie_id": 11,
    "title": "Savannah Run",
    "Action": 0,
    "Comedy": 1,
    "Drama": 1,
    "Romance": 0,
    "SciFi": 0
}
movies = pd.concat([movies, pd.DataFrame([new_movie])], ignore_index=True)
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
0,1,Galaxy Mission,1,0,0,0,1,0.857143
1,2,Love in Nairobi,0,0,1,1,0,0.142857
2,3,Final Battle,1,0,0,0,0,0.500000
3,4,Family Weekend,0,1,0,0,0,0.000000
4,5,Future World,1,0,0,0,1,0.857143
5,6,Broken Hearts,0,0,1,1,0,0.142857
6,7,City Detectives,1,0,1,0,0,0.571429
7,8,Space Warriors,1,0,0,0,1,0.857143
8,9,Wedding Weekend,0,1,0,1,0,0.071429
9,10,Hidden Truth,0,0,1,0,0,0.071429


## 21. Create Your Own User (Step 25)

I created a user who loves Action and SciFi, moderately likes Comedy, and
dislikes Romance. Ratings are on at least four movies.

In [ ]:
my_user = pd.DataFrame({
    "movie_id": [1, 3, 6, 9],
    "rating": [5, 4, 2, 3]
})
recommend_movies(movies, my_user, features, n=5)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,0.583333
7,8,Space Warriors,1,0,0,0,1,0.583333
6,7,City Detectives,1,0,1,0,0,0.458333
1,2,Love in Nairobi,0,0,1,1,0,0.291667
10,11,Savannah Run,0,1,1,0,0,0.208333


## 22. Interpretation Questions (Step 26)

1. **Two strongest preferences:** **Action (0.375)** and **SciFi (0.208)**,
   driven by the 5-star *Galaxy Mission* (Action + SciFi) and the 4-star
   *Final Battle* (Action). Romance also reaches 0.208, but only from the
   lower 2- and 3-star ratings.
2. **Three highest-scoring movies:** *Future World* (0.583), *Space Warriors*
   (0.583), and *City Detectives* (0.458).
3. **Why did the top movie rank first?** *Future World* and *Space Warriors*
   tie because both contain exactly the user's two strongest features
   (Action + SciFi), so both capture the largest share of the normalized
   profile weight.
4. **Any surprises?** *Savannah Run* (my added movie) appears in the Top 5
   even though this user rated only one Comedy/Drama movie, and low. Because
   the model treats genres independently and additively, a Comedy+Drama movie
   still collects small weights — it ranks last of the five, but it enters the
   list once the obvious Action/SciFi titles are exhausted. This shows the
   model recommends by feature matching alone, with no sense of overall
   quality.
5. **What important information is the model ignoring?** It ignores *which*
   aspects of a movie drove the rating (acting, plot, director), the strength
   of dislike (a 2-star and a 3-star non-Action film contribute similarly),
   other users' ratings entirely (no collaborative signal), popularity, and
   metadata such as year, language, cast and description.

## 23. Limitations (Step 27)

1. **Only five coarse features** — one-hot genres lose all nuance (no sub-genres,
   themes, mood, or quality signals), so many different movies look identical to
   the model.
2. **Rich metadata ignored** — actors, directors, release year, language and
   descriptions are unused, so two films with the same genres but totally
   different appeal are indistinguishable.
3. **Over-specialization (filter bubble)** — recommendations are always
   feature-matches to what the user already liked, so the user keeps getting
   more of the same and never discovers genuinely different content.
4. **New-user cold start** — with few or no ratings (like User C above), the
   profile is empty or unreliable, so recommendations are poor until the user
   supplies history.
5. **Garbage-in, garbage-out metadata** — if the genre tags are wrong or
   incomplete, the profiles, user models and scores inherit those errors
   directly.
6. **No collaborative signal** — the model cannot learn from what *similar
   users* enjoyed; it only knows this user's own history.

## 24. Conclusion

This lab built a complete content-based filtering pipeline from structured
movie features. I created item profiles as binary genre vectors, derived a
normalized user profile by rating-weighting the features of watched movies,
scored every candidate movie against that profile, removed already-watched
items, and produced Top-N recommendations. Testing Users A and B showed the
model personalizes recommendations, unlike Week 2's popularity baseline.
However, the approach depends entirely on the quality and breadth of features
and ratings, suffers from cold start and over-specialization, and will be
extended in Week 4 with TF-IDF text features and cosine similarity.

**Final question:** If two users watched exactly the same movies but gave them
different ratings, should this recommender produce the same recommendations?
**No.** The user profile is built by *weighting each movie's features by its
rating* before summing. Different ratings change every profile weight: a user
who rated the Action films 5 stars and the Drama film 2 stars gets an
Action-heavy profile, while a user who rated them 1 star and 5 stars
respectively gets a Drama-heavy profile. Since recommendation scores are
computed from each user's own normalized profile, the same movies receive
different scores and therefore different Top-N lists. The viewing history
determines *which* features are possible; the ratings determine *how strongly*
each feature is preferred.